# E4.1 — DSBPSO vs Standard BPSO

Compares DSBPSO stickiness with standard BPSO variants in terms of solution quality and search behaviour.

**Experiment:** 3 algorithm configurations × N seeds, using best encoding (E1.1), best CHT (E2.1), and best valuation (E3.1).

| Config | Algorithm | Inertia Weight |
|--------|-----------|----------------|
| dsbpso | DSBPSO (stickiness) | N/A (uses i_s/i_p/i_g) |
| bpso_constant_w | Standard BPSO | Constant w=0.9 |
| bpso_increasing_w | Standard BPSO | Increasing 0.4→0.9 |

**Metrics:** HVI, IGD, convergence curves, search dynamics (Hamming distance, per-variable-type entropy).

In [ ]:
import sys
sys.path.insert(0, "..")

import pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from itertools import combinations
from scipy.stats import wilcoxon

def paired_wilcoxon(metric_a, seeds_a, metric_b, seeds_b):
    """Wilcoxon signed-rank paired by seed. Returns (p_value, n_common)."""
    seed_to_a = dict(zip(seeds_a, metric_a))
    seed_to_b = dict(zip(seeds_b, metric_b))
    common = sorted(set(seed_to_a) & set(seed_to_b))
    va = np.array([seed_to_a[s] for s in common])
    vb = np.array([seed_to_b[s] for s in common])
    if np.allclose(va, vb):
        return 1.0, len(common)
    return wilcoxon(va, vb, alternative="two-sided").pvalue, len(common)


def paired_diff(metric_a, seeds_a, metric_b, seeds_b):
    """Per-seed (A - B) differences, aligned by seed."""
    sa_to_v = dict(zip(seeds_a, metric_a))
    sb_to_v = dict(zip(seeds_b, metric_b))
    common = sorted(set(sa_to_v) & set(sb_to_v))
    return np.array([sa_to_v[s] - sb_to_v[s] for s in common])

from src.experiments.runner import load_experiment_results, load_run_result
from src.experiments.metrics import (
    hypervolume_2d,
    inverted_generational_distance,
    construct_reference_front,
)

config_order = ["dsbpso", "bpso_constant_w", "bpso_increasing_w"]
config_labels = ["DSBPSO", "BPSO (w=0.9)", "BPSO (w: 0.4→0.9)"]
colors = ["steelblue", "coral", "seagreen"]

## 1. Load Results

In [ ]:
runs = load_experiment_results("E4.1")
print(f"Loaded {len(runs)} runs")

configs = defaultdict(list)
for run in runs:
    s = run["summary"]
    if s["status"] != "completed":
        print(f"  SKIPPING failed run: {s['run_id']}")
        continue
    configs[s["config_name"]].append(run)

for name, run_list in sorted(configs.items()):
    times = [r["summary"]["elapsed_seconds"] for r in run_list]
    print(f"  {name:20s}: {len(run_list)} runs, avg time {np.mean(times):.1f}s")

## 2. HVI and IGD

In [ ]:
fronts_by_config = {}
for name, run_list in configs.items():
    fronts = []
    for run in run_list:
        result = load_run_result(run["pickle_path"])
        pf = result["pareto_front"]
        if pf:
            obj = np.array([e.get("objectives_final", e["objectives"]) for e in pf])
            fronts.append(obj)
        else:
            fronts.append(np.empty((0, 2)))
    fronts_by_config[name] = fronts

all_objectives = [f for fronts in fronts_by_config.values() for f in fronts if len(f) > 0]
reference_front = construct_reference_front(all_objectives)
all_combined = np.vstack(all_objectives)
ref_point = np.array([
    all_combined[:, 0].min() - abs(all_combined[:, 0].min()) * 0.1,
    all_combined[:, 1].min() - abs(all_combined[:, 1].min()) * 0.1,
])

seeds_by_config = {name: [r["summary"]["seed"] for r in run_list] for name, run_list in configs.items()}
hvi_by_config = {}
igd_by_config = {}
for name, fronts in fronts_by_config.items():
    hvi_by_config[name] = np.array([hypervolume_2d(f, ref_point) for f in fronts])
    igd_by_config[name] = np.array([inverted_generational_distance(f, reference_front) for f in fronts])

print(f"{'Config':20s} {'HVI mean':>12s} {'HVI std':>10s} {'IGD mean':>12s} {'IGD std':>10s}")
print("-" * 66)
for name in config_order:
    if name not in hvi_by_config: continue
    hvi, igd = hvi_by_config[name], igd_by_config[name]
    print(f"{name:20s} {hvi.mean():12.1f} {hvi.std():10.1f} {igd.mean():12.4f} {igd.std():10.4f}")

## 3. Box Plots — HVI and IGD

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
active = [c for c in config_order if c in hvi_by_config]
labels = [config_labels[config_order.index(c)] for c in active]

for ax, data_dict, ylabel, title in [
    (axes[0], {c: hvi_by_config[c] / 1e6 for c in active}, r"HVI ($\times 10^6$)", "Hypervolume Indicator (higher = better)"),
    (axes[1], igd_by_config, "IGD", "Inverted Generational Distance (lower = better)"),
]:
    data = [data_dict[c] for c in active]
    bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.6)
    for p, color in zip(bp["boxes"], colors): p.set_facecolor(color); p.set_alpha(0.7)
    ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xlim(0.4, len(active) + 0.6)

plt.tight_layout()
plt.show()

## 4. Statistical Significance — Pairwise

In [ ]:
active = [c for c in config_order if c in hvi_by_config]
n_configs = len(active)
n_comparisons = n_configs * (n_configs - 1) // 2
bonferroni_alpha = 0.05 / max(1, n_comparisons)

print(f"Pairwise Wilcoxon signed-rank, paired by seed (Bonferroni alpha = {bonferroni_alpha:.4f}):\n")
for i, j in combinations(range(n_configs), 2):
    ci, cj = active[i], active[j]
    li = config_labels[config_order.index(ci)]
    lj = config_labels[config_order.index(cj)]
    p_hvi, _ = paired_wilcoxon(hvi_by_config[ci], seeds_by_config[ci], hvi_by_config[cj], seeds_by_config[cj])
    p_igd, _ = paired_wilcoxon(igd_by_config[ci], seeds_by_config[ci], igd_by_config[cj], seeds_by_config[cj])
    sig_h = "***" if p_hvi < 0.001 else "**" if p_hvi < 0.01 else "*" if p_hvi < bonferroni_alpha else "n.s."
    sig_i = "***" if p_igd < 0.001 else "**" if p_igd < 0.01 else "*" if p_igd < bonferroni_alpha else "n.s."
    print(f"{li} vs {lj}:")
    print(f"  HVI: p = {p_hvi:.6f} {sig_h}")
    print(f"  IGD: p = {p_igd:.6f} {sig_i}\n")

## 5. BPSO Inertia Weight Selection

Compare the two BPSO variants to select the better baseline for the main DSBPSO vs BPSO comparison.

In [ ]:
bpso_configs = [c for c in ["bpso_constant_w", "bpso_increasing_w"] if c in hvi_by_config]
if len(bpso_configs) == 2:
    c1, c2 = bpso_configs
    hvi_1, hvi_2 = hvi_by_config[c1].mean(), hvi_by_config[c2].mean()
    best_bpso = c1 if hvi_1 >= hvi_2 else c2
    best_bpso_label = config_labels[config_order.index(best_bpso)]
    print(f"Best BPSO variant by HVI: {best_bpso_label}")
    print(f"  {config_labels[config_order.index(c1)]}: HVI = {hvi_1:.1f}")
    print(f"  {config_labels[config_order.index(c2)]}: HVI = {hvi_2:.1f}")

    # Main comparison: DSBPSO vs best BPSO
    if "dsbpso" in hvi_by_config:
        p, _ = paired_wilcoxon(hvi_by_config["dsbpso"], seeds_by_config["dsbpso"], hvi_by_config[best_bpso], seeds_by_config[best_bpso])
        print(f"\nDSBPSO vs {best_bpso_label} (main comparison):")
        print(f"  HVI: DSBPSO = {hvi_by_config['dsbpso'].mean():.1f}, {best_bpso_label} = {hvi_by_config[best_bpso].mean():.1f}")
        print(f"  p = {p:.6f}")

## 6. Overlaid Pareto Fronts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for i, name in enumerate(config_order):
    if name not in hvi_by_config: continue
    hvi_vals = hvi_by_config[name]
    median_idx = np.argmin(np.abs(hvi_vals - np.median(hvi_vals)))
    front = fronts_by_config[name][median_idx]
    if len(front) == 0: continue
    ax.scatter(front[:, 1], front[:, 0], color=colors[i], label=config_labels[i], s=30, alpha=0.8)

ax.set_xlabel("CVaR (MNOK)"); ax.set_ylabel("Expected ROV (MNOK)")
ax.set_title("Pareto Fronts — Median-HVI Run Per Algorithm")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Convergence Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, name in enumerate(config_order):
    if name not in configs: continue
    all_rov, all_cvar = [], []
    for run in configs[name]:
        hp = run["run_dir"] / "history.pkl"
        if hp.exists():
            with open(hp, "rb") as f: h = pickle.load(f)
            all_rov.append(h["best_rov"]); all_cvar.append(h["best_cvar"])
    if not all_rov: continue
    max_len = max(len(r) for r in all_rov)
    rov_arr = np.full((len(all_rov), max_len), np.nan)
    cvar_arr = np.full((len(all_cvar), max_len), np.nan)
    for j, (r, c) in enumerate(zip(all_rov, all_cvar)):
        rov_arr[j, :len(r)] = r; cvar_arr[j, :len(c)] = c
    iters = np.arange(1, max_len + 1)
    axes[0].plot(iters, np.nanmean(rov_arr, axis=0), color=colors[i], label=config_labels[i], alpha=0.8)
    axes[1].plot(iters, np.nanmean(cvar_arr, axis=0), color=colors[i], label=config_labels[i], alpha=0.8)

axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("Best E[ROV] (MNOK)")
axes[0].set_title("Convergence — Best Expected ROV"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel("Iteration"); axes[1].set_ylabel("Best CVaR (MNOK)")
axes[1].set_title("Convergence — Best CVaR"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Search Dynamics — Swarm Diversity

Average pairwise Hamming distance over iterations. Higher = more diverse swarm.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
for i, name in enumerate(config_order):
    if name not in configs: continue
    all_hamming = []
    for run in configs[name]:
        hp = run["run_dir"] / "history.pkl"
        if hp.exists():
            with open(hp, "rb") as f: h = pickle.load(f)
            if "hamming_distance" in h:
                all_hamming.append(h["hamming_distance"])
    if not all_hamming: continue
    max_len = max(len(hd) for hd in all_hamming)
    arr = np.full((len(all_hamming), max_len), np.nan)
    for j, hd in enumerate(all_hamming): arr[j, :len(hd)] = hd
    iters = np.arange(1, max_len + 1)
    mean = np.nanmean(arr, axis=0)
    std = np.nanstd(arr, axis=0)
    ax.plot(iters, mean, color=colors[i], label=config_labels[i], linewidth=2, alpha=0.9)
    ax.fill_between(iters, mean - std, mean + std, color=colors[i], alpha=0.15)

ax.set_xlabel("Iteration"); ax.set_ylabel("Avg pairwise Hamming distance (bits)")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Search Dynamics — Per-Variable-Type Entropy

Binary entropy per variable type (assignment, timing, wells) over iterations. Shows whether different variable types converge at different rates.

In [ ]:
entropy_keys = ["entropy_assignment", "entropy_timing", "entropy_wells"]
entropy_labels = ["Assignment", "Timing", "Wells"]
entropy_colors = ["steelblue", "coral", "seagreen"]

fig, axes = plt.subplots(1, len(config_order), figsize=(6 * len(config_order), 5), sharey=True)
if not hasattr(axes, '__len__'): axes = [axes]

for ax_idx, name in enumerate(config_order):
    if name not in configs: continue
    ax = axes[ax_idx]

    for k, (ekey, elabel, ec) in enumerate(zip(entropy_keys, entropy_labels, entropy_colors)):
        all_ent = []
        for run in configs[name]:
            hp = run["run_dir"] / "history.pkl"
            if hp.exists():
                with open(hp, "rb") as f: h = pickle.load(f)
                if ekey in h: all_ent.append(h[ekey])
        if not all_ent: continue
        max_len = max(len(e) for e in all_ent)
        arr = np.full((len(all_ent), max_len), np.nan)
        for j, e in enumerate(all_ent): arr[j, :len(e)] = e
        iters = np.arange(1, max_len + 1)
        ax.plot(iters, np.nanmean(arr, axis=0), color=ec, label=elabel, linewidth=2, alpha=0.9)

    ax.set_xlabel("Iteration")
    ax.set_title(config_labels[config_order.index(name)])
    ax.legend()

axes[0].set_ylabel("Binary entropy")
plt.tight_layout()
plt.show()

## 10. Summary and Algorithm Selection

Selects the algorithm for the final configuration used in E5.

In [ ]:
active = [c for c in config_order if c in hvi_by_config]
print("Ranking by mean HVI (higher = better):")
for rank, (name, val) in enumerate(sorted([(n, hvi_by_config[n].mean()) for n in active], key=lambda x: -x[1]), 1):
    print(f"  {rank}. {config_labels[config_order.index(name)]:25s} HVI = {val:.1f}")

print("\nRanking by mean IGD (lower = better):")
for rank, (name, val) in enumerate(sorted([(n, igd_by_config[n].mean()) for n in active], key=lambda x: x[1]), 1):
    print(f"  {rank}. {config_labels[config_order.index(name)]:25s} IGD = {val:.4f}")

best = sorted([(n, hvi_by_config[n].mean()) for n in active], key=lambda x: -x[1])[0][0]
print(f"\nSelected algorithm for E5: {config_labels[config_order.index(best)]}")